# 01 — Verify the uploaded data before spending GPU on it

Run this **once per Kaggle dataset version**, before any training notebook.

It re-derives every hash in `provenance.json` from the files actually attached to this
session. That is not paranoia: this study has already lost a day to arms built from the
wrong corpus, and the failure was invisible to every automated gate. A stale or partial
upload fails here in 30 seconds instead of after 36 runs.

**Attach:** `emocap-v2-arms` (required). `emocap-v2-classifier` is optional here and only
needed to score — attach it and the last cell will verify it too.

The two datasets are split by *when* they are needed — arms and features to train, the
classifier to score — because a single 421 MB upload stalled for 54 minutes and created
nothing. Both carry the same commit and the same classifier hash, so the halves can always
be checked against each other.

In [ ]:
import hashlib, json, os
from pathlib import Path

DATA = Path("/kaggle/input/emocap-v2-arms")
prov = json.loads((DATA / "provenance.json").read_text())
print("commit ", prov["git_commit"][:12])
print("tag    ", prov["prereg_tag"])
print("clean  ", prov["tree_clean"])
print("clf    ", prov["classifier_sha256"][:16])
assert prov["tree_clean"], "uploaded from a dirty tree -- it matches no commit"

In [ ]:
def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

bad, checked = [], 0
for rel, want in prov["files"].items():
    p = DATA / rel
    if not p.exists():
        bad.append((rel, "MISSING")); continue
    got = sha256(p)
    checked += 1
    if got != want:
        bad.append((rel, f"{got[:12]} != {want[:12]}"))

print(f"verified {checked}/{len(prov['files'])} files")
for rel, why in bad:
    print("  BAD", rel, why)
assert not bad, "upload does not match its manifest -- re-push before training"

In [ ]:
# Arm counts must match the manifest, which must match the registration.
man = prov["arm_manifest"]
print(f"{'arm':<13}{'cells':>9}{'images':>8}  per-register")
for arm, m in man["arms"].items():
    n = sorted(set(m["per_register"].values()))
    lines = sum(1 for _ in open(DATA / "arms" / f"{arm}.jsonl"))
    assert lines == m["cells"], f"{arm}: {lines} lines but manifest says {m['cells']}"
    print(f"{arm:<13}{m['cells']:>9,}{m['images']:>8,}  {n}")
print("\nselection rules:")
for k, v in man["selection"].items():
    print(f"  {k}: {v}")

In [ ]:
# Features must cover every image every arm references. An uncovered image would
# otherwise surface as a KeyError thirty minutes into a run.
import numpy as np

idx = set(json.loads((DATA / "features" / "clip_vit_b32_index.json").read_text()))
arr = np.load(DATA / "features" / "clip_vit_b32.npy", mmap_mode="r")
print("features", arr.shape, arr.dtype)
for arm in man["arms"]:
    ids = {json.loads(l)["image_id"] for l in open(DATA / "arms" / f"{arm}.jsonl")}
    gap = ids - idx
    print(f"  {arm:<13} {len(ids):>6,} images  uncovered {len(gap)}")
    assert not gap, f"{arm} references {len(gap)} images with no features"

# The classifier lives in its own dataset. If it is attached, check that it is the SAME
# instrument this data version was registered against -- pairing arms with a differently
# trained classifier would make every number incomparable, silently.
CLF = Path("/kaggle/input/emocap-v2-classifier")
if CLF.exists():
    cprov = json.loads((CLF / "provenance.json").read_text())
    assert cprov["git_commit"] == prov["git_commit"], (
        f"classifier built at {cprov['git_commit'][:8]} but data at "
        f"{prov['git_commit'][:8]} -- these two halves are not from one commit")
    on_disk = (CLF / "classifier" / "sha256.txt").read_text().strip()
    assert on_disk == prov["classifier_sha256"], "classifier hash differs from the manifest"
    print(f"\nclassifier attached and matched: {on_disk[:16]}")
else:
    print("\nclassifier dataset not attached (only needed to score)")

print("\nALL CHECKS PASSED -- safe to train")